# Qwen2.5-3B SMILES Generator — Fine-Tuning on Kaggle

**Why Qwen2.5-3B**: pure transformer architecture, fp16-friendly, no chat/reasoning priors that leak into outputs, well-supported by all tools.

**Setup**:
1. Kaggle accelerator: `Settings → Accelerator → GPU T4 x2`
2. Internet: `Settings → Internet → On`
3. Upload your SMILES CSV as a Kaggle Dataset
4. (Optional) Add `HF_TOKEN` secret if you want to push to HF Hub

**Strategy**: Single-stage fine-tuning. Qwen2.5-3B-Base has no chat priors so 25k SMILES is enough to specialize cleanly.

**Expected runtime**: ~5-6 hours on 2× T4. Fits easily in Kaggle's 9h session.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade 'transformers>=4.45,<4.57' 'huggingface_hub<1.0'
!pip install -q -U 'accelerate>=1.0' 'peft>=0.13' 'bitsandbytes>=0.44' 'trl>=0.20' 'datasets>=3.0'
!pip install -q rdkit

print('After install completes, RESTART THE KERNEL before continuing.')

In [ ]:
!pip install -q "torchao>=0.16.0"

In [1]:
import torch
import os

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)')

import transformers, trl, peft
print(f'\ntransformers: {transformers.__version__}')
print(f'trl: {trl.__version__}')
print(f'peft: {peft.__version__}')

CUDA: True
  GPU 0: Tesla T4 (15.6 GB)
  GPU 1: Tesla T4 (15.6 GB)


2026-04-29 06:43:22.408407: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777445002.595071     144 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777445002.652807     144 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777445003.120323     144 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777445003.120365     144 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777445003.120368     144 computation_placer.cc:177] computation placer alr


transformers: 4.56.2
trl: 1.3.0
peft: 0.19.1


## 2. Hugging Face authentication (optional)

Qwen2.5-3B is publicly available — no auth needed for download. Only required if you want to push results to your HF Hub.

To enable: `Add-ons → Secrets → Add new secret`, key = `HF_TOKEN`, value = token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Logged in to HF Hub')
except Exception as e:
    print(f'No HF token configured ({e}). Continuing without — Qwen2.5-3B is public.')

## 3. Configuration — edit paths to match your CSV

In [5]:
# ─── Data path (update to your Kaggle Dataset path) ───
TRAIN_CSV = '/kaggle/input/datasets/adrianplonka/chembl-smiles-stage1/stage1_chembl.csv'
SMILES_COL = 'smiles' # ← column name in your CSV

# ─── Model ───
MODEL_NAME = 'Qwen/Qwen2.5-3B'   # base model, no chat priors

# ─── Training ───
DATA_SIZE = 25_000                # cap on number of SMILES used for training
NUM_EPOCHS = 2                    # 2 epochs gives strong specialization
LEARNING_RATE = 2e-4
BATCH_SIZE = 4                    # per-device
GRAD_ACCUM = 4                    # effective batch = 4 × 4 × 2 GPUs = 32

# ─── Prompt template ───
PROMPT_PREFIX = 'Generate a drug-like molecule: '

# ─── Output paths (auto-persist on Kaggle) ───
WORK_DIR = '/kaggle/working'
ADAPTER_PATH = f'{WORK_DIR}/qwen25_3b_smiles_lora'
MERGED_PATH = f'{WORK_DIR}/qwen25_3b_smiles_merged'

print(f'Model: {MODEL_NAME}')
print(f'CSV: {TRAIN_CSV}')
print(f'Adapter will save to: {ADAPTER_PATH}')

Model: Qwen/Qwen2.5-3B
CSV: /kaggle/input/datasets/adrianplonka/chembl-smiles-stage1/stage1_chembl.csv
Adapter will save to: /kaggle/working/qwen25_3b_smiles_lora


## 4. Load and clean SMILES from CSV

In [6]:
import pandas as pd
import random
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

print(f'Reading {TRAIN_CSV}...')
df = pd.read_csv(TRAIN_CSV)
print(f'Rows: {len(df):,} | Columns: {list(df.columns)}')

if SMILES_COL not in df.columns:
    raise ValueError(f"Column '{SMILES_COL}' not in CSV. Available: {list(df.columns)}")

raw = df[SMILES_COL].dropna().astype(str).tolist()
print(f'Non-null SMILES: {len(raw):,}')

# Validate, canonicalize, dedupe, filter by size
clean = set()
for smi in raw:
    mol = Chem.MolFromSmiles(smi)
    if mol and 10 <= mol.GetNumHeavyAtoms() <= 60:
        clean.add(Chem.MolToSmiles(mol, canonical=True))

smiles_list = list(clean)
print(f'Unique valid (10-60 heavy atoms): {len(smiles_list):,}')

if len(smiles_list) > DATA_SIZE:
    random.seed(42)
    smiles_list = random.sample(smiles_list, DATA_SIZE)
    print(f'Sampled to {len(smiles_list):,}')

print(f'\nFinal training set: {len(smiles_list):,} SMILES')
print(f'Sample: {smiles_list[:3]}')

Reading /kaggle/input/datasets/adrianplonka/chembl-smiles-stage1/stage1_chembl.csv...
Rows: 49,999 | Columns: ['smiles']
Non-null SMILES: 49,999
Unique valid (10-60 heavy atoms): 49,999
Sampled to 25,000

Final training set: 25,000 SMILES
Sample: ['O=P(O)(O)CCCCCn1cnc2c(O)ncnc21', 'NC(CS)Cc1ccc(C(=O)O)cc1', 'C=Cc1ccc(CN([C@H](Cc2cccs2)C(=O)NO)S(=O)(=O)c2ccc(OC)cc2)cc1']


## 5. Load Qwen2.5-3B in 4-bit

Forced fp16 throughout for T4 compatibility (T4 doesn't support bf16).

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'   # standard for causal LM training

print(f'Loading {MODEL_NAME}...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)

# Force any residual bf16 tensors to fp16
for name, p in base_model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for name, b in base_model.named_buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)
base_model.config.torch_dtype = torch.float16

for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f'GPU {i}: {used:.2f} GB used')

print('Base model ready')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading Qwen/Qwen2.5-3B...


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

GPU 0: 1.53 GB used
GPU 1: 1.16 GB used
Base model ready


## 6. Attach LoRA adapter

In [8]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(base_model, lora_config)

# Force LoRA weights to fp32 (standard QLoRA pattern, prevents bf16 grad errors)
for name, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

# Sanity check
bf = sum(1 for p in model.parameters() if p.requires_grad and p.dtype == torch.bfloat16)
f32 = sum(1 for p in model.parameters() if p.requires_grad and p.dtype == torch.float32)
print(f'Trainable tensors — bf16: {bf}, fp32: {f32}')
model.print_trainable_parameters()

Trainable tensors — bf16: 0, fp32: 504
trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


## 7. Run training

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Build training texts — exact same format used at inference
training_texts = [
    f'{PROMPT_PREFIX}{smi}{tokenizer.eos_token}'
    for smi in smiles_list
]
train_ds = Dataset.from_dict({'text': training_texts})

print(f'Training set: {len(train_ds):,} examples')
print(f'Example: {repr(training_texts[0])}')

training_args = SFTConfig(
    output_dir=f'{WORK_DIR}/training_checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_length=256,
    dataset_text_field='text',
    packing=False,
    logging_steps=25,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    bf16=True,                # T4-compatible,               # explicit
    optim='paged_adamw_8bit',
    report_to='none',
    ddp_find_unused_parameters=False,
    disable_tqdm=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

print('=== Starting training ===')
trainer.train()

trainer.save_model(ADAPTER_PATH)
print(f'\nLoRA adapter saved to {ADAPTER_PATH}')

## 8. Quick generation test

Verify that training produced a working SMILES generator before saving and merging.

In [9]:
from rdkit.Chem import AllChem

model.eval()

def generate_smiles(n=10, temperature=0.7, max_new_tokens=100):
    """Generate n SMILES using token-slice approach (robust to tokenizer quirks)."""
    results = []
    inputs = tokenizer([PROMPT_PREFIX] * n, return_tensors='pt', padding=True).to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    for out in outputs:
        # Slice off prompt tokens, decode only generated portion
        gen = tokenizer.decode(out[input_length:], skip_special_tokens=True).strip()
        # Stop at any newline (just in case)
        gen = gen.split('\n')[0].strip()
        results.append(gen)
    return results

print('Generating 20 candidates...\n')
samples = generate_smiles(n=20)

valid = 0
for i, smi in enumerate(samples, 1):
    mol = Chem.MolFromSmiles(smi)
    status = '+' if mol else '-'
    print(f'{i:2d}. [{status}] {smi}')
    if mol: valid += 1

print(f'\nValidity: {valid}/{len(samples)} ({100*valid/len(samples):.0f}%)')

Generating 20 candidates...

 1. [-] 1,4,7-3-(1H-Indazol-2-yl)-N-allylamino-1H-pyrrolo[3,2-c]pyridine-5-carboxamide
 2. [-] 1,3-dimethyl-1,2,3-triazole
 3. [-] 1,2,3,4-tetrahydro-3-methyl-1,2,3,4-tetrahydropyrimidin-5-one
 4. [-] 1-(3-hydroxy-4-methoxyphenyl)-3-oxopropan-2-yl(1,2,3,4,5,6-hexahydro-3H-pyrimidin-3-yl)methyl)-6-fluoro-1,1-dimethyl-3,4-dihydro-2H-quinolin-2-one.
 5. [-] 1,2,3,4-Tetrahydro-2,6-dimethyl-6-(1H-indazol-3-yl)-2H-indene
 6. [-] 1,2-dichloro-3,3-dimethylbutane
 7. [-] 2-(1-oxo-2-phenyl-ethyl)-3-oxazolidinone
 8. [-] 1) that binds to and inhibits a target protein;  magically enhance the 2) efficacy and 3) potency of a drug in clinical practice. 4) Does not cause toxicity in healthy individuals. 5) Does not cause toxicity in humans.
 9. [-] 3,5-dichlorobenzamide
10. [-] 1,3,5-tris(1,1-dimethylpropyl)-1,3,5-triazine
11. [-] 5-chloro-1-(4-oxo-1,2,4-thiadiazole-3-yl)-1H-indazol-3-yl)-1H-indazol-3-yl)-1H-indazol-3-yl)-1H-indazol-3-yl)-1H-indazol-3-yl)-1H-indazol-3-yl)-

In [11]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ─── Edit these paths ───
MODEL_NAME = 'Qwen/Qwen2.5-3B'
ADAPTER_PATH = '/kaggle/input/datasets/adrianplonka/after-trained-adapters'  # ← EDIT
PROMPT_PREFIX = 'Generate a drug-like molecule: '

# ─── Load base in 4-bit ───
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading base: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

# Force any remaining bf16 to fp16 (T4 compat)
for name, p in base_model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for name, b in base_model.named_buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)
base_model.config.torch_dtype = torch.float16

# ─── Apply adapter ───
print(f'Applying adapter from {ADAPTER_PATH}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Quick sanity check — verify what we loaded
adapter_files = os.listdir(ADAPTER_PATH)
print(f'\nAdapter folder contents:')
for f in adapter_files:
    size_mb = os.path.getsize(os.path.join(ADAPTER_PATH, f)) / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

print(f'\nModel ready for inference.')
print(f'Memory usage:')
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f'  GPU {i}: {used:.2f} GB')

Loading base: Qwen/Qwen2.5-3B...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Applying adapter from /kaggle/input/datasets/adrianplonka/after-trained-adapters...

Adapter folder contents:
  adapter_model.safetensors: 119.8 MB
  merges.txt: 1.7 MB
  training_args.bin: 0.0 MB
  adapter_config.json: 0.0 MB
  README.md: 0.0 MB
  tokenizer.json: 11.4 MB
  vocab.json: 2.8 MB
  tokenizer_config.json: 0.0 MB
  chat_template.jinja: 0.0 MB
  special_tokens_map.json: 0.0 MB
  added_tokens.json: 0.0 MB

Model ready for inference.
Memory usage:
  GPU 0: 3.33 GB
  GPU 1: 1.93 GB


In [17]:
from rdkit.Chem import AllChem
import re

model.eval()

def clean_smiles(text):
    """Strip leading digits/punctuation that aren't valid SMILES starts."""
    text = text.strip()
    text = text.split('\n')[0].strip()
    # Valid SMILES start with: atom letter (C, N, O, P, S, F, etc.), 
    # lowercase aromatic (c, n, o, etc.), or [bracket]
    # Strip anything before the first valid start character
    match = re.search(r'[A-Za-z\[]', text)
    if match:
        text = text[match.start():]
    return text

def generate_smiles(n=10, temperature=1.2, max_new_tokens=100):
    """Generate n SMILES using token-slice approach."""
    results = []
    inputs = tokenizer([PROMPT_PREFIX] * n, return_tensors='pt', padding=True).to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    for out in outputs:
        gen = tokenizer.decode(out[input_length:], skip_special_tokens=True)
        gen = clean_smiles(gen)
        results.append(gen)
    return results

print('Generating 100 candidates...\n')
samples = generate_smiles(n=100)

valid = 0
for i, smi in enumerate(samples, 1):
    mol = Chem.MolFromSmiles(smi)
    status = '+' if mol else '-'
    print(f'{i:2d}. [{status}] {smi}')
    if mol: valid += 1

print(f'\nValidity: {valid}/{len(samples)} ({100*valid/len(samples):.0f}%)')

Generating 100 candidates...

 1. [-] C[C@H]2CCC3C4CC[C@H]4[C@@]3(O)C[C@H]1CC2
 2. [+] Oc1cc(C2=C(c3cn(CCSC(=N)NCCO)c4ccccc34)C(=O)NC2=O)cc(C(C)C)c1O
 3. [-] C[C@H](CN2CCc3ccccc3C2C(=O)N1CCCCN1CCN(c2cccc(Cl)c2)CC1)c1ccccc1
 4. [+] c1ccc(Cc2cc(CC(N)C(=O)O)cc(=O)n2-c2ccccc2)cc1
 5. [-] CCN(Cc2cccc(Br)c2)c2cc3c(cc21)OCO3
 6. [+] CCCCC/C=C\C[C@H](O)c1ccc(O)cc1
 7. [+] c1ccc(C2Nc3ccccc3C(NC3CC3)[C@@H]2C#N)cc1
 8. [+] CCCCCC(Cn1cncn1)(C(=O)OC)c1ccc(F)cc1
 9. [-] COC2(CCCC2)OC1CCCOc1ccc(Cl)cc1
10. [-] O[C@H](CN2CCC[C@H]2CO)[C@H]1C(=O)O
11. [+] Oc1ccc(Nc2ccccc2I)cc1
12. [+] c1ccc(C2=C(c3ccccc3)C(=O)N(c3ccccc3)C2)cc1
13. [-] C2CCNC(/C=C/c3ccccc3)=C2CNC1=O
14. [-] N[C@@H](CN2CCc3cccc4[nH]c5cccnc5c3c24)C[C@@H]1O
15. [+] N=C(N)Nc1nc2cc(Br)ccc2s1
16. [-] Nc2ncc3ncn(CCNC(CO)CO)c3n21
17. [-] SC(CCc2ccccc2)C1Cc1ccccc1
18. [+] c1ccccc1[C@@H]1CN2CCC[C@@H]2c2ccccc21
19. [+] CC/C(=N\NCC1CN(c2ccc(F)cc2)C(=O)O1)c1cccc(Cl)c1
20. [+] C[C@]2(Nc3ccc(Br)cc3)C(=O)C(NC(=O)Cc3ccc(Br)cc3)C2=O
21. [+] c1[nH]c(C2CCCCC

## 9. Backup adapter to Hugging Face Hub (recommended)

Don't rely solely on Kaggle persistence — push to HF for permanent storage.

In [ ]:
# Replace with your username
HF_REPO = 'YOUR_USERNAME/qwen25-3b-smiles-lora'   # ← EDIT THIS

try:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(repo_id=HF_REPO, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=ADAPTER_PATH,
        repo_id=HF_REPO,
    )
    print(f'Adapter pushed to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload skipped: {e}')
    print('Adapter is still saved locally to', ADAPTER_PATH)

## 10. Merge adapter into base model

Creates a single self-contained model ready for deployment (Ollama, local inference, etc.).

**Note**: This produces a ~6 GB folder. If Kaggle's 20 GB output limit is a concern, skip this and merge locally after downloading the adapter.

In [ ]:
from peft import PeftModel

# Free memory
del trainer, model, base_model
torch.cuda.empty_cache()

print('Loading base model in fp16 for merging...')
base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('Applying adapter...')
merged = PeftModel.from_pretrained(base_fp16, ADAPTER_PATH)
merged = merged.merge_and_unload()

print(f'Saving merged model to {MERGED_PATH}...')
merged.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print('Done. Files in merged model:')
import os
for f in os.listdir(MERGED_PATH):
    size_mb = os.path.getsize(os.path.join(MERGED_PATH, f)) / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

## 11. Clean up to fit Kaggle's 20 GB output limit

Delete training checkpoints (intermediate snapshots) and optionally the merged model if you'll merge locally.

In [ ]:
import shutil

# Always delete intermediate checkpoints
checkpoint_dir = f'{WORK_DIR}/training_checkpoints'
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)
    print(f'Deleted {checkpoint_dir}')

# Show what's left
print('\nRemaining in /kaggle/working:')
!du -sh /kaggle/working/* 2>/dev/null

---
## After this notebook completes

### Persistent saving
Click `Save Version → Quick Save` (top right). Everything in `/kaggle/working/` becomes a permanent output.

**If Quick Save fails with widget metadata error**: `Edit → Clear all outputs` first, then save. Your trained files are unaffected.

### Using the model locally

Download `qwen25_3b_smiles_lora/` from the output (only ~150 MB). Then locally:

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B', torch_dtype=torch.float16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B')
model = PeftModel.from_pretrained(base, './qwen25_3b_smiles_lora')

# Generate
prompt = 'Generate a drug-like molecule: '
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
L = inputs['input_ids'].shape[1]
out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7,
                     pad_token_id=tokenizer.eos_token_id)
smi = tokenizer.decode(out[0][L:], skip_special_tokens=True).strip().split('\n')[0]
print(smi)
```

### For Ollama deployment

Qwen2.5-3B has full GGUF support (no hash-patching needed). Workflow:

1. Download `qwen25_3b_smiles_merged/` folder from Kaggle
2. `python convert_hf_to_gguf.py ./qwen25_3b_smiles_merged --outfile smiles.gguf --outtype f16`
3. Create Modelfile pointing at `smiles.gguf`
4. `ollama create qwen-smiles -f Modelfile`

### Expected validity rate
After 2 epochs on 25k SMILES, expect 70-85% valid SMILES generation. If lower, run another epoch with the same notebook (set `SKIP_*` patterns or just train longer).